# Register and Deploy Investment Copilot Agent

Logs the LangGraph agent to MLflow, registers it in Unity Catalog,
and creates a Model Serving endpoint.

**Prerequisites:**
- Gold tables exist (pipeline has run)
- A SQL warehouse is available for the agent's query tool

In [ ]:
%pip install -U mlflow>=2.12.0 langchain-databricks>=0.1.0 langgraph>=0.2.0 databricks-sdk>=0.81.0
dbutils.library.restartPython()

In [ ]:
import mlflow
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint, DatabricksSQLWarehouse

CATALOG = "startups_catalog"
SCHEMA = "gold"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.investment_copilot"
ENDPOINT_NAME = "investment_copilot"

mlflow.set_registry_uri("databricks-uc")
print(f"Model: {MODEL_NAME}")
print(f"Endpoint: {ENDPOINT_NAME}")

## Log Agent to MLflow

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Find a SQL warehouse for the agent
warehouses = list(w.warehouses.list())
warehouse_id = warehouses[0].id if warehouses else None
print(f"SQL warehouse: {warehouse_id}")

# Define resources the agent needs access to
resources = [
    DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-4"),
]
if warehouse_id:
    resources.append(DatabricksSQLWarehouse(warehouse_id=warehouse_id))

In [ ]:
# Agent code is in the bundle's deployed files
import os

# When running via DABs job, the working dir is the bundle root
agent_code_path = os.path.join(os.getcwd(), "src", "agent", "agent.py")

# If running from notebooks directory, adjust
if not os.path.exists(agent_code_path):
    # Try relative to bundle files location
    bundle_root = "/Workspace" + os.environ.get("DATABRICKS_BUNDLE_PATH", "")
    agent_code_path = os.path.join(bundle_root, "files", "src", "agent", "agent.py")

print(f"Agent code path: {agent_code_path}")
print(f"Exists: {os.path.exists(agent_code_path)}")

In [ ]:
input_example = {
    "messages": [
        {"role": "user", "content": "What is the average occupancy rate across all properties?"}
    ]
}

with mlflow.start_run(run_name="investment_copilot"):
    model_info = mlflow.langchain.log_model(
        lc_model=agent_code_path,
        artifact_path="agent",
        input_example=input_example,
        registered_model_name=MODEL_NAME,
        resources=resources,
        pip_requirements=[
            "mlflow>=2.12.0",
            "langchain-databricks>=0.1.0",
            "langgraph>=0.2.0",
            "databricks-sdk>=0.81.0",
        ],
    )

print(f"Model logged: {model_info.model_uri}")

## Deploy to Serving Endpoint

In [ ]:
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AutoCaptureConfigInput,
)

# Get latest model version
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(v.version for v in versions)
print(f"Latest version: {latest_version}")

# Check if endpoint exists
endpoint_exists = False
try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    endpoint_exists = True
    print(f"Endpoint '{ENDPOINT_NAME}' exists — will update.")
except Exception:
    print(f"Endpoint '{ENDPOINT_NAME}' not found — will create.")

In [ ]:
served_entities = [
    ServedEntityInput(
        entity_name=MODEL_NAME,
        entity_version=str(latest_version),
        scale_to_zero_enabled=True,
    )
]

if endpoint_exists:
    print(f"Updating endpoint '{ENDPOINT_NAME}'...")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=served_entities,
    )
else:
    print(f"Creating endpoint '{ENDPOINT_NAME}'...")
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            served_entities=served_entities,
            auto_capture_config=AutoCaptureConfigInput(
                catalog_name=CATALOG,
                schema_name=SCHEMA,
                enabled=True,
            ),
        ),
    )

print(f"Endpoint '{ENDPOINT_NAME}' is ready.")

In [ ]:
# Quick test
print("Testing endpoint...")
response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    messages=[{"role": "user", "content": "How many properties are in the portfolio?"}],
)
print(f"Response: {response.choices[0].message.content[:200]}")